In [1]:
!pip install torch torchaudio librosa sounddevice scikit-learn

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNAudioGRU(nn.Module):
    def __init__(self, num_classes, input_channels=1):
        super(CNNAudioGRU, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.5)
        self.gru_input_size = 1024
        self.gru = nn.GRU(
            input_size=self.gru_input_size,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.5
        )
        self.attention = nn.Linear(512, 1)
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()
        x = x.view(b, w, c*h)
        x, _ = self.gru(x)
        attn_weights = F.softmax(self.attention(x), dim=1)
        x = torch.sum(x * attn_weights, dim=1)
        x = self.fc(x)
        return x


In [3]:
import pickle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 31  # Change if your number of classes is different

# Load model
model = CNNAudioGRU(num_classes=NUM_CLASSES)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()

# Load label encoder
with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)


/Users/keshavgogia/Desktop/Speech Intent Recoginition/.venv/lib/python3.11/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.5.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
import sounddevice as sd
from scipy.io.wavfile import write

def record_audio(filename="user_recording.wav", duration=2, fs=16000):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    write(filename, fs, audio)
    print(f"Audio saved to {filename}")
    return filename


In [5]:
import librosa
import numpy as np

def extract_melspectrogram(file_path, sr=16000, n_mels=64, duration=2.0):
    y, _ = librosa.load(file_path, sr=sr, duration=duration)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - np.mean(mel_db)) / (np.std(mel_db) + 1e-6)
    return mel_db.astype(np.float32)


In [6]:
def predict_intent(audio_path, model, le, device, sr=16000, n_mels=64, duration=2.0):
    mel_db = extract_melspectrogram(audio_path, sr=sr, n_mels=n_mels, duration=duration)
    mel_tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)  # [1, n_mels, time]
    mel_tensor = mel_tensor.to(device)
    with torch.no_grad():
        output = model(mel_tensor)
        pred_idx = output.argmax(dim=1).cpu().item()
        intent = le.inverse_transform([pred_idx])[0]
    return intent


In [13]:
# Step 1: Record user's speech
audio_file = record_audio(filename="user_recording.wav", duration=2, fs=16000)

# Step 2: Predict intent
intent = predict_intent(audio_file, model, le, device)
print("Predicted intent:", intent)


Recording for 2 seconds...
Audio saved to user_recording.wav
Predicted intent: decrease_volume_none
